# E8 (v2) — Per-class F1 for the 5 classifiers — pipeline COPIED VERBATIM from the training notebooks

**Lesson from v1 (every sanity check FAILED):** the crop set and preprocessing must be IDENTICAL to training, not merely "equivalent".
v2 copies verbatim from notebooks 01/02/03:

1. **build_crops** exactly as written: `pad=0.15` around the box, drop boxes with `w<16 or h<16` px, cap `MAX_CROPS_PER_CLASS=2500`
   (not applied to test), iterate files `sorted` then `random.Random(42).shuffle`, crop at size **128** (default PIL resize).
2. **VGG16/ResNet50**: `keras.applications.<backbone>.preprocess_input(X.astype("float32"))` — caffe mode, NOT /255.
3. **GRU**: `X/255.` then reshape to `(N, 128, 384)` — as in notebook 02.
4. **ViT-B/S**: from the 128 px uint8 crop → `cv2.resize` to 224 (bilinear) → `/255` → ImageNet normalisation (0.485/0.229) — exactly notebook 03's CropDS.
5. **Metrics** use the notebooks' own `multiclass_metrics` function (worst-F1 excludes classes with support<10).

**Mandatory sanity check:** accuracy/macro-F1 must match the published figures (ViT-B 0.9521/0.8711; ViT-S 0.9453/0.8429;
ResNet50 0.9014/0.8072; VGG16 0.8966/0.7894; GRU-256 0.8075/0.6780) within ±0.5 accuracy points / ±2 F1 points.
If everything matches, the per-class F1 values are valid for the report.

**Output:** `results/e8_perclass_f1.csv`, `results/e8_confusion_<model>.png`, `results/e8_summary_check.csv`.

**Add Input:** `ppe-unified-v1` + `ppe-trained-weights`. GPU T4 recommended (~20 min; CPU ~1-2 h, ViT slowest).

In [ ]:
KEY_PREFIX = "e8_"
# ==================== CONFIG (the exact constants used in training) ====================
QUICK_TEST = False
SEED = 42
CROP_SIZE, CROP_SIZE_VIT = 128, 224
MAX_CROPS_PER_CLASS = 200 if QUICK_TEST else 2500

import os, random, shutil, glob, json, time
WORK = "/kaggle/working"; os.makedirs(f"{WORK}/results", exist_ok=True)
STATE_PATH = f"{WORK}/state.json"
STATE = {}
for _st in glob.glob("/kaggle/input/**/state.json", recursive=True):
    try: _s = json.load(open(_st))
    except Exception: continue
    if any(k.startswith(KEY_PREFIX) and isinstance(_s[k], dict) and _s[k].get("v") == 2 for k in _s):
        STATE.update(_s); print("Resumed state (v2):", list(_s))
def save_state(): json.dump(STATE, open(STATE_PATH, "w"), ensure_ascii=False, indent=1)
def done(key): return key in STATE
random.seed(SEED)


In [ ]:
%pip install -q timm
import torch, numpy as np, cv2
np.random.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (works, but ViT will be slow)")


In [ ]:
from pathlib import Path
import yaml
def find_dataset_root():
    for c in glob.glob("/kaggle/input/**/data.yaml", recursive=True):
        root = os.path.dirname(c)
        if all(os.path.isdir(f"{root}/{s}/images") for s in ("train", "val", "test")):
            return root
    raise AssertionError("Dataset ppe-unified-v1 has not been Added as Input!")
DATA = find_dataset_root()
CLASSES = yaml.safe_load(open(f"{DATA}/data.yaml"))["names"]
if isinstance(CLASSES, dict): CLASSES = [CLASSES[k] for k in sorted(CLASSES)]
NUM_CLS = len(CLASSES)
print("Dataset:", DATA, "|", NUM_CLS, "classes")


In [ ]:
# ---------- build_crops: COPIED VERBATIM from notebooks 01/02/03 (cell 5) ----------
from collections import Counter
from PIL import Image

def build_crops(split, size):
    X, y = [], []; per_class = Counter()
    files = sorted(os.listdir(f"{DATA}/{split}/labels"))
    random.Random(SEED).shuffle(files)
    for lf in files:
        stem = Path(lf).stem
        if "_rfs" in stem: continue
        for ext in [".png", ".jpg", ".jpeg"]:
            ip = f"{DATA}/{split}/images/{stem}{ext}"
            if os.path.exists(ip): break
        else: continue
        img = Image.open(ip).convert("RGB"); W, H = img.size
        for line in open(f"{DATA}/{split}/labels/{lf}"):
            if not line.strip(): continue
            c, xc, yc, bw, bh = map(float, line.split()); c = int(c)
            if per_class[c] >= MAX_CROPS_PER_CLASS: continue
            w, h = bw * W, bh * H
            if w < 16 or h < 16: continue
            pad = 0.15
            box = (max(0, (xc - bw/2) * W - pad * w), max(0, (yc - bh/2) * H - pad * h),
                   min(W, (xc + bw/2) * W + pad * w), min(H, (yc + bh/2) * H + pad * h))
            X.append(np.asarray(img.crop(box).resize((size, size)), dtype=np.uint8)); y.append(c)
            per_class[c] += 1
    return np.array(X), np.array(y)

Xte, yte = build_crops("test", CROP_SIZE)
print("Crop test:", Xte.shape, {CLASSES[i]: int(v) for i, v in enumerate(np.bincount(yte, minlength=NUM_CLS))})

# ---------- multiclass_metrics: verbatim from the training notebooks ----------
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
def multiclass_metrics(y_true, pred):
    acc = accuracy_score(y_true, pred)
    macro = f1_score(y_true, pred, average="macro")
    per_cls = f1_score(y_true, pred, average=None, labels=list(range(NUM_CLS)), zero_division=0)
    support = np.bincount(y_true, minlength=NUM_CLS)
    valid = [c for c in range(NUM_CLS) if support[c] >= 10]
    worst = float(np.min([per_cls[c] for c in valid])) if valid else 0.0
    return acc, macro, worst


In [ ]:
# ---------- Locate the weights ----------
def find_one(patterns):
    for pat in patterns:
        hits = sorted(glob.glob(f"/kaggle/input/**/{pat}", recursive=True))
        if hits: return hits[0]
    return None

WEIGHTS = {
    "ViT-B/16": find_one(["vit_base*best*.pt"]),
    "ViT-S/16": find_one(["vit_small*best*.pt"]),
    "ResNet50": find_one(["resnet50*best*.keras"]),
    "VGG16":    find_one(["vgg16*best*.keras"]),
    "GRU-256":  find_one(["rnn_best*.keras", "gru*best*.keras"]),
}
for k, v in WEIGHTS.items(): print(f"{k:10s} -> {v}")
PUBLISHED = {"ViT-B/16": (0.9521, 0.8711), "ViT-S/16": (0.9453, 0.8429),
             "ResNet50": (0.9014, 0.8072), "VGG16": (0.8966, 0.7894), "GRU-256": (0.8075, 0.6780)}


In [ ]:
# ---------- Predict using EXACTLY each branch's preprocessing ----------
import pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt

def predict_keras_cnn(path, backbone):
    """VGG16/ResNet50: caffe-mode preprocess_input — VERBATIM from notebook 01."""
    import tensorflow as tf
    from tensorflow import keras
    pp = {"VGG16": keras.applications.vgg16.preprocess_input,
          "ResNet50": keras.applications.resnet50.preprocess_input}[backbone]
    model = keras.models.load_model(path, compile=False)
    return model.predict(pp(Xte.astype("float32")), batch_size=64, verbose=0).argmax(1)

def predict_gru(path):
    """GRU: /255 then reshape to (N, 128, 384) — VERBATIM from notebook 02."""
    from tensorflow import keras
    Xte_s = (Xte.astype("float32") / 255.).reshape(len(Xte), CROP_SIZE, CROP_SIZE * 3)
    model = keras.models.load_model(path, compile=False)
    return model.predict(Xte_s, batch_size=64, verbose=0).argmax(1)

def predict_vit(path, arch):
    """ViT: 128 px uint8 crop -> cv2.resize 224 -> /255 -> ImageNet normalisation — VERBATIM from notebook 03's CropDS."""
    import timm
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    model = timm.create_model(arch, num_classes=NUM_CLS)
    model.load_state_dict(torch.load(path, map_location=dev)); model.to(dev).eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(Xte), 64):
            batch = []
            for img in Xte[i:i+64]:
                t = torch.from_numpy(cv2.resize(img, (CROP_SIZE_VIT, CROP_SIZE_VIT))).permute(2, 0, 1).float() / 255.
                batch.append((t - MEAN) / STD)
            preds.append(model(torch.stack(batch).to(dev)).argmax(1).cpu().numpy())
    return np.concatenate(preds)

rows, checks = [], []
for name, path in WEIGHTS.items():
    if path is None: print("SKIPPED (weights missing):", name); continue
    if done(f"e8_{name}"): print("skip:", name); continue
    t0 = time.time()
    if name == "ViT-B/16": y_pred = predict_vit(path, "vit_base_patch16_224")
    elif name == "ViT-S/16": y_pred = predict_vit(path, "vit_small_patch16_224")
    elif name == "GRU-256": y_pred = predict_gru(path)
    else: y_pred = predict_keras_cnn(path, name)
    acc, mf1, worst = multiclass_metrics(yte, y_pred)
    pa, pf = PUBLISHED[name]
    ok = abs(acc - pa) < 0.005 and abs(mf1 - pf) < 0.02
    print(f"{name}: acc {acc:.4f} (published {pa}) | macro-F1 {mf1:.4f} ({pf}) | worst {worst:.4f} -> {'MATCH ✓' if ok else 'MISMATCH ✗'}")
    checks.append({"model": name, "acc": round(acc, 4), "acc_published": pa,
                   "macro_f1": round(mf1, 4), "f1_published": pf, "worst_f1": round(worst, 4), "sanity_ok": ok,
                   "min": round((time.time() - t0) / 60, 1)})
    rep = classification_report(yte, y_pred, labels=list(range(NUM_CLS)), target_names=CLASSES,
                                output_dict=True, zero_division=0)
    for cls in CLASSES:
        rows.append({"model": name, "class": cls,
                     **{k: round(rep[cls][k], 4) for k in ("precision", "recall", "f1-score")},
                     "support": int(rep[cls]["support"]), "sanity_ok": ok})
    cm = confusion_matrix(yte, y_pred, labels=list(range(NUM_CLS)), normalize="true")
    fig, ax = plt.subplots(figsize=(9, 8)); im_ = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLS)); ax.set_xticklabels(CLASSES, rotation=90, fontsize=7)
    ax.set_yticks(range(NUM_CLS)); ax.set_yticklabels(CLASSES, fontsize=7)
    ax.set_title(f"Confusion matrix (normalized) — {name} (test)"); fig.colorbar(im_)
    fig.savefig(f"{WORK}/results/e8_confusion_{name.replace('/', '_').replace('-', '_')}.png",
                dpi=120, bbox_inches="tight"); plt.close(fig)
    STATE[f"e8_{name}"] = {"v": 2, "acc": round(acc, 4), "macro_f1": round(mf1, 4), "sanity_ok": ok}; save_state()

pd.DataFrame(rows).to_csv(f"{WORK}/results/e8_perclass_f1.csv", index=False)
pd.DataFrame(checks).to_csv(f"{WORK}/results/e8_summary_check.csv", index=False)
print(pd.DataFrame(checks).to_string(index=False))


In [ ]:
# ==================== CLEAN UP OUTPUT ====================
KEEP = {"results", "state.json"}
for item in os.listdir(WORK):
    if item in KEEP: continue
    p = os.path.join(WORK, item)
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)
print("Final output:", sorted(os.listdir(WORK)))
